# Bundled reconstruction: cleaning, synthesis and ablation

This notebook runs a **small, source-separated reconstruction** from the eight bundled breathing sources. It is not the historical training corpus and does not establish clinical performance.


In [ ]:
!git clone --depth 1 https://github.com/Chaitanya701402/airway-signal-lab.git airway-signal-lab
%cd airway-signal-lab
!pip -q install 'tensorflow>=2.16,<2.19' librosa==0.11.0 scipy scikit-learn matplotlib soundfile
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## Add the reconstruction scripts

Upload `experiments/build_bundled_reconstruction.py` and `experiments/plot_bundled_results.py` from the submitted workspace. The builder validates compressed payloads, fixes participant splits before deriving windows, synthesizes the three constructed classes, and writes a source-group manifest.


In [ ]:
from google.colab import files
uploaded = files.upload()
required = {'build_bundled_reconstruction.py', 'plot_bundled_results.py'}
assert required <= set(uploaded), f'Upload: {required}'
!mkdir -p experiments
!mv build_bundled_reconstruction.py experiments/build_bundled_reconstruction.py
!mv plot_bundled_results.py experiments/plot_bundled_results.py
!python experiments/build_bundled_reconstruction.py


In [ ]:
import json, pandas as pd
root = 'experiments/bundled_reconstruction'
display(pd.read_csv(f'{root}/window_manifest.csv').groupby(['split','class_name']).size().unstack(fill_value=0))
print(json.dumps(json.load(open(f'{root}/dataset_metadata.json')), indent=2))


## Train the controlled comparison

E0 is CNN-only, E1 adds a GRU, E2 adds temporal attention, and E3 uses both. Each model uses the same frozen source split, seed and validation-loss checkpointing.


In [ ]:
!python experiments/hybrid_ablation.py --data {root}/prepared_source_split.npz --manifest {root}/window_manifest.csv --output {root}/ablation --epochs 35 --batch-size 16
display(pd.read_csv(f'{root}/ablation/ablation_results.csv'))


In [ ]:
!python experiments/plot_bundled_results.py --experiment-dir {root}
from IPython.display import Image, display
display(Image(f'{root}/pipeline_overview.png'))
display(Image(f'{root}/ablation_comparison.png'))
display(Image(f'{root}/confusion_matrix_E2.png'))
display(Image(f'{root}/ablation/training_curves.png'))
!zip -r bundled_reconstruction_results.zip {root}/ablation {root}/dataset_metadata.json {root}/window_manifest.csv {root}/*.png
files.download('bundled_reconstruction_results.zip')
